# Planes de ahorro con InvestingLab

Esta demo simula un plan de ahorro sobre una cartera: capital inicial, aportación mensual y un
objetivo en dinero de hoy. Calcula la probabilidad de alcanzarlo, cuánto habría que aportar
para llegar con más seguridad y cuánto dura un capital del que se retira dinero cada mes.

La cartera se simula entera, mes a mes, a partir de su valor histórico. Más detalle en
`docs/guide/planning.md`.

In [ ]:
import pandas as pd

from investinglab.config import LogLevel, configure_logging
from investinglab.domain import (
    DriftAssumption,
    SavingsPlan,
    SimulationConfig,
    VolatilityAssumption,
)
from investinglab.toolkit import Toolkit

configure_logging(LogLevel.WARNING)
tk = Toolkit.from_settings()

SIMBOLOS = ["SAN.MC", "ITX.MC", "IBE.MC", "BBVA.MC"]
cartera = tk.portfolio("Núcleo", dict.fromkeys(SIMBOLOS, 0.25), start="2019-01-01")
SIMULACION = SimulationConfig(n_paths=5_000)

## Un plan con objetivo

10 000 € al empezar, 300 € al final de cada mes (un 2 % más cada año) y el objetivo de llegar
a 150 000 € de hoy en 20 años, con una inflación del 2 %.

In [ ]:
plan = SavingsPlan(
    years=20,
    initial=10_000,
    contribution=300,
    contribution_growth=0.02,
    goal=150_000,
    inflation=0.02,
)
resultado = cartera.savings_plan(plan, SIMULACION)
resultado.summary()

In [ ]:
resultado.plot()

## ¿Cuánto hay que aportar?

La aportación mensual que alcanza el objetivo en cada proporción de escenarios. Es exacta: el
patrimonio final crece en línea recta con la aportación, así que no hace falta volver a
simular.

In [ ]:
aportaciones = pd.Series(
    {
        f"{probabilidad:.0%}": resultado.required_contribution(probabilidad)
        for probabilidad in (0.5, 0.75, 0.9, 0.95)
    },
    name="Aportación mensual necesaria",
)
aportaciones

## Con supuestos prudentes

La rentabilidad histórica de pocos años es muy ruidosa. Con una rentabilidad esperada del 5 %
y una volatilidad del 15 % fijadas a mano, el resultado ya no depende de la suerte del
histórico.

In [ ]:
prudente = SimulationConfig(
    n_paths=5_000,
    drift=DriftAssumption.ASSUMED,
    expected_returns=0.05,
    volatility=VolatilityAssumption.ASSUMED,
    volatilities=0.15,
)
con_supuestos = cartera.savings_plan(plan, prudente)
pd.Series(
    {
        "Histórico": resultado.probability_of_success,
        "Supuestos prudentes": con_supuestos.probability_of_success,
    },
    name="Probabilidad de alcanzar el objetivo",
)

## Retiradas: ¿cuánto dura el dinero?

Una aportación negativa es una retirada. Con 400 000 € y 1 800 € al mes durante 30 años, ¿en
qué proporción de escenarios se acaba el dinero antes?

In [ ]:
jubilacion = cartera.savings_plan(
    SavingsPlan(years=30, initial=400_000, contribution=-1_800, inflation=0.02), prudente
)
jubilacion.probability_of_ruin

In [ ]:
tk.close()